# 04 — Signal hunt

**Given a field that varies, what does it actually mean?**

Fit a candidate field against a known ground truth: wall-clock time, a
multimeter reading, counted wheel rotations, a lever position. A field that
fits nothing measured stays a hypothesis, however plausible the curve looks.

In [ ]:
import sys
from pathlib import Path

# The project is not installed as a package, so put the repo root on the path.
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO))

import pandas as pd
pd.set_option("display.width", 200, "display.max_columns", 40)

from bikecan import dbc, discover, experiment, ids, session

SESSIONS = REPO / "data" / "sessions"

In [ ]:
s = session.load(SESSIONS / "legacy")
TARGET = "04FF3400"   # change this to the identifier you are working on

discover.byte_detail(s.can, TARGET)

## Try every plausible width and endianness

The same bytes read as 8, 16 or 32 bits, big or little endian, give very
different curves. Only one of them is usually physically plausible.

In [ ]:
import numpy as np

frames = s.can[s.can["id_hex"] == TARGET].sort_values("timestamp")
payloads = [bytes(p) for p in frames["data"]]
timestamps = frames["timestamp"].to_numpy()

def field(start_byte, width, endian):
    chunk = [p[start_byte:start_byte + width] for p in payloads]
    if any(len(c) < width for c in chunk):
        return None
    return np.array([int.from_bytes(c, endian) for c in chunk], dtype=float)

results = []
for width in (1, 2, 3, 4):
    for start in range(0, 8 - width + 1):
        for endian in ("big", "little"):
            values = field(start, width, endian)
            if values is None or np.unique(values).size < 3:
                continue
            elapsed = timestamps - timestamps[0]
            # A monotonic field that tracks elapsed time is a counter or a clock.
            monotonic = bool(np.all(np.diff(values) >= 0))
            slope = np.polyfit(elapsed, values, 1)[0] if elapsed.max() > 0 else np.nan
            results.append({
                "bytes": f"{start}..{start + width - 1}",
                "width_bits": width * 8,
                "endian": endian,
                "distinct": int(np.unique(values).size),
                "min": values.min(),
                "max": values.max(),
                "monotonic": monotonic,
                "per_second": round(slope, 5),
                "seconds_per_count": round(1 / slope, 3) if slope else np.nan,
            })

pd.DataFrame(results).sort_values(["monotonic", "distinct"], ascending=False)

## Fit against ground truth

For the legacy corpus the ground truth available is wall-clock time, which is
what confirmed the uptime counter: the field advances by exactly one count per
10 seconds of elapsed time, and it holds across the gaps between recordings.

For a physical signal, replace `elapsed` below with the measured quantity —
speed from counted wheel rotations, pack voltage from a multimeter, throttle
from lever position.

In [ ]:
values = field(2, 2, "big")
elapsed = timestamps - timestamps[0]

deltas_value = np.diff(values)
deltas_time = np.diff(timestamps)
ratio = deltas_time / deltas_value

print(f"counts: {values.min():.0f} to {values.max():.0f}")
print(f"seconds per count: min {ratio.min():.4f}  max {ratio.max():.4f}  "
      f"median {np.median(ratio):.4f}")
print(f"exact at every step: {bool(np.allclose(ratio, np.median(ratio), atol=0.05))}")

In [ ]:
# A field that tracks a real quantity looks like that quantity over time.
import plotly.express as px

frame = pd.DataFrame({"elapsed_s": elapsed, "value": values})
figure = px.line(frame, x="elapsed_s", y="value", markers=True,
                 title=f"{TARGET} bytes 2-3, big endian")
figure.update_layout(height=400, xaxis_title="elapsed (s)", yaxis_title="raw value")
figure.show()

## Record it

A fit is not a finding until it is written down with its evidence. Add the
signal to `dbc/signals.toml` with an honest confidence level:

- `hypothesis` — consistent with the recordings, no stimulus test
- `probable` — a stimulus experiment supports it, nothing measured
- `confirmed` — fitted against a measured value

Then regenerate and re-run the tests:

```
uv run python -m bikecan.dbc
uv run pytest -q
```